# LISS-IV Preprocessing Pipeline — Demo Notebook

This notebook walks through every step of the preprocessing and multi-temporal fusion pipeline,
showing intermediate outputs for each stage.


In [ ]:
import sys
sys.path.insert(0, '../..')  # repo root

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import rasterio
from rasterio.transform import from_bounds

## 1. Generate Synthetic Test Data

In [ ]:
def create_test_tif(path, seed=0, h=256, w=256, bands=3):
    rng = np.random.default_rng(seed)
    data = (rng.random((bands, h, w)) * 255).astype(np.uint8)
    transform = from_bounds(77.0, 28.0, 77.1, 28.1, w, h)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        path, 'w', driver='GTiff',
        height=h, width=w, count=bands, dtype='uint8',
        crs='EPSG:4326', transform=transform
    ) as dst:
        dst.write(data)
    print(f'Created: {path}')

create_test_tif('../../backend/data/current/lissiv_current.tif', seed=42)
create_test_tif('../../backend/data/historical/20230101.tif', seed=10)
create_test_tif('../../backend/data/historical/20230601.tif', seed=20)
print('Test data ready.')

## 2. Load Image with Metadata

In [ ]:
from backend.preprocessing.utils import load_image

current_data, current_meta = load_image('../../backend/data/current/lissiv_current.tif')
print('Shape:', current_data.shape)
print('CRS:', current_meta['crs'])
print('Resolution:', current_meta['resolution'])
print('Dtype:', current_meta['dtype'])

plt.figure(figsize=(5,5))
plt.imshow(current_data[:,:,:3].astype(float) / 255)
plt.title('Current Image (raw DN)')
plt.axis('off')
plt.show()

## 3. Radiometric Calibration

In [ ]:
from backend.preprocessing.calibration import RadiometricCalibrator

calibrator = RadiometricCalibrator(noise_method='gaussian', noise_sigma=1.0)
current_calib = calibrator.calibrate(current_data, current_meta)
print('Calibrated range:', current_calib.min(), '-', current_calib.max())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(current_data[:,:,:3].astype(float)/255); axes[0].set_title('Before calibration'); axes[0].axis('off')
axes[1].imshow(current_calib[:,:,:3]); axes[1].set_title('After calibration'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 4. Patch Selector — Choose Best Historical Image

In [ ]:
from backend.preprocessing.patch_selector import PatchSelector
from datetime import date

selector = PatchSelector()
best_path, scores = selector.select_best(
    current_calib,
    '../../backend/data/historical/',
    current_date=date(2023, 6, 15)
)
print('Best historical image:', best_path.name)
for name, s in scores.items():
    print(f'  {name}: total={s["total"]:.3f}')

## 5. Geometric Alignment

In [ ]:
from backend.preprocessing.utils import load_image
from backend.preprocessing.alignment import GeometricAligner

hist_data, hist_meta = load_image(best_path)
hist_calib = calibrator.calibrate(hist_data, hist_meta, reference=current_calib)

aligner = GeometricAligner(detector='orb', ecc_fallback=True)
aligned, reg_error = aligner.align(hist_calib, current_calib)
print(f'Registration error: {reg_error:.3f} px')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(current_calib[:,:,:3]); axes[0].set_title('Current'); axes[0].axis('off')
axes[1].imshow(hist_calib[:,:,:3]); axes[1].set_title('Historical (unaligned)'); axes[1].axis('off')
axes[2].imshow(aligned[:,:,:3]); axes[2].set_title(f'Aligned (err={reg_error:.2f}px)'); axes[2].axis('off')
plt.tight_layout(); plt.show()

## 6. Normalisation

In [ ]:
from backend.preprocessing.normalization import BandNormalizer

normalizer = BandNormalizer(method='minmax')
current_norm = normalizer.normalise(current_calib)
historical_norm = normalizer.normalise(aligned, stats=normalizer.band_stats)
print('Normalised range:', current_norm.min(), '-', current_norm.max())

## 7. Feature Fusion

In [ ]:
from backend.preprocessing.fusion import FeatureFusion

fusion = FeatureFusion(method='channel_stack')
fused = fusion.fuse(current_norm, historical_norm)
print('Fused shape:', fused.shape)

plt.figure(figsize=(5,5))
plt.imshow(fused[:,:,:3])
plt.title('Fused Image (first 3 channels)')
plt.axis('off')
plt.show()

## 8. Full Pipeline (one-liner)

In [ ]:
from backend.preprocessing.pipeline import PreprocessingPipeline
from datetime import date

pipeline = PreprocessingPipeline('../../backend/configs/pipeline_config.yaml')
result = pipeline.run(
    current_path='../../backend/data/current/lissiv_current.tif',
    historical_dir='../../backend/data/historical/',
    current_date=date(2023, 6, 15),
    output_dir='../../backend/data/output/demo',
)

print('Processing time:', result.metadata['processing_time'], 's')
print('Fused shape:', result.fused_image.shape)
print('Output files:')
for k, v in result.output_paths.items():
    print(f'  {k}: {v}')

## 9. Preview Images

In [ ]:
import cv2

def show_preview(path, title):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(5,5))
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')
    plt.show()

for label, path in result.preview_paths.items():
    show_preview(path, label)